# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. We use unique `@id`s to reference each entity in the schema, ensuring clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
meta = dataset.metadata
print(f"Name: {getattr(meta, 'name', None)}\n\nDescription: {getattr(meta, 'description', None)}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

In [ ]:
# Find all RecordSets and print their @id, name, and field structure

record_sets = list(dataset.metadata.record_sets)
print(f"Number of Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Select all available record_set @id's
all_record_sets = [rs.id for rs in dataset.metadata.record_sets]
print("RecordSets available:", all_record_sets)

dataframes = {}
for record_set_id in all_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Pick the first record set as an example
main_rs_id = all_record_sets[0]
print(f"\nColumns in RecordSet '{main_rs_id}':\n", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Filter records and perform simple processing using field `@id`s.

In [ ]:
# Choose a numeric field for demonstration
# We'll search for the first numeric field in the selected record set
main_rs = next(rs for rs in dataset.metadata.record_sets if rs.id == main_rs_id)
numeric_field_id = None
for field in main_rs.fields:
    if getattr(field, 'data_type', '').lower() in ['integer', 'float', 'number']:
        numeric_field_id = field.id
        break
if not numeric_field_id:
    raise ValueError('No numeric field found for this demonstration.')

print(f"Using numeric field: {numeric_field_id}")

# Filter records with example threshold
threshold = 10
df = dataframes[main_rs_id]

# If the field is loaded as object, try to convert
if df[numeric_field_id].dtype == object:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (showing top rows):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by the first non-numeric field
group_field_id = None
for field in main_rs.fields:
    dt = getattr(field, 'data_type', '').lower()
    if dt not in ['integer', 'float', 'number']:
        group_field_id = field.id
        break
if group_field_id and group_field_id in df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped by '{group_field_id}' (average {numeric_field_id}):")
    print(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram for the numeric field after normalization
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
plt.title(f"Distribution of Normalized Field: {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Frequency")
plt.show()

# If grouping succeeded, plot group means
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    grouped.plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load and explore a dataset defined by a Croissant schema using `mlcroissant`.
- Access record sets, fields, and their unique `@id`s.
- Extract and process data from record sets into DataFrames.
- Perform simple filtering, normalization, and grouping for EDA.
- Visualize field distributions and aggregate group statistics.

**Next steps:** For deeper analysis, you might select specific fields or record sets, perform domain-specific transformations, or integrate the data with your downstream ML or analytics pipelines. Always refer to the schema `@id` for reproducibility!